## Cross-Architecture Comparison: JEPA vs Supervised

Loads precomputed results from `scripts/analyze_comparison.py` and produces:
1. Effective dimensionality bar chart (JEPA vs supervised)
2. SAE feature overlap cosine histogram
3. Velocity and curvature distribution overlays
4. Prospective probe AUROC grouped bar chart per label
5. Label subspace alignment heatmap

In [ ]:
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.use("module://matplotlib_inline.backend_inline")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

-- Config --

In [ ]:
JEPA_EXP     = "stopg_42_v01"
SUP_EXP      = "supervised_64_42"
EXPERIMENTS  = Path("experiments")
JEPA_DIR     = EXPERIMENTS / JEPA_EXP
RESULTS_DIR  = JEPA_DIR / "results"
FIGURES_DIR  = RESULTS_DIR / "comparison" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

-- Load results --

In [ ]:
with open(RESULTS_DIR / "comparison.json") as f:
    results = json.load(f)

npz = dict(np.load(RESULTS_DIR / "comparison.npz", allow_pickle=True))

print(f"Matched samples: {results['n_matched']}")
print(f"CKA: {results['cka']:.4f}")

### 1. Effective Dimensionality

Compares the intrinsic dimensionality of JEPA vs supervised representations
using PCA-based metrics: effective dimensionality (Shannon entropy of
normalized eigenvalues), and the number of PCs needed for 90% / 95% variance.

In [ ]:
from src.analysis.plotting import plot_effective_dim_comparison, show_or_savefig

plot_effective_dim_comparison(
    results["pca_stats"]["jepa"],
    results["pca_stats"]["supervised"],
    show=True,
    save_path=FIGURES_DIR / "effective_dim_comparison",
)

### 2. SAE Feature Overlap

Hungarian matching of decoder directions between the JEPA SAE and
supervised SAE. The histogram shows per-feature cosine similarity;
features above the threshold are considered "stable" across architectures.


In [ ]:
from src.analysis.plotting import plot_sae_overlap_histogram

if "sae_matched_cosines" in npz:
    plot_sae_overlap_histogram(
        npz["sae_matched_cosines"],
        show=True,
        save_path=FIGURES_DIR / "sae_overlap_histogram",
    )
else:
    print("No SAE overlap data (run with --jepa-sae and --sup-sae)")


### 3. Velocity and Curvature Distributions

Compares how patient trajectories move through representation space.
Velocity = step-to-step displacement magnitude, curvature = cosine angle
between successive velocity vectors. Higher velocity = more dynamic
representations; curvature near 0 = straight-line trajectories.

In [ ]:
from src.analysis.plotting import plot_trajectory_distributions

plot_trajectory_distributions(
    npz["jepa_velocities"],
    npz["sup_velocities"],
    npz["jepa_curvatures"],
    npz["sup_curvatures"],
    show=True,
    save_path=FIGURES_DIR / "trajectory_distributions",
)

### 4. Prospective Probe AUROC

Causal test: can trajectory features (velocity toward concept centroid)
predict future labels better than static embeddings? Groups show
JEPA trajectory vs baseline and supervised trajectory vs baseline.

In [ ]:
from src.analysis.plotting import plot_probe_auroc_comparison

probe_comparison = results.get("probe_comparison", {})
if probe_comparison:
    plot_probe_auroc_comparison(
        probe_comparison,
        show=True,
        save_path=FIGURES_DIR / "probe_auroc_comparison",
    )
else:
    print("No probe comparison results found")

### 5. Label Subspace Alignment Heatmap

Entry (i, j) = mean cosine of principal angles between label i's subspace
in JEPA and label j's subspace in supervised. High diagonal values mean
both architectures encode the same concept in the same directions. Off-diagonal
values reveal cross-label contamination or shared structure.

In [ ]:
from src.analysis.plotting import plot_label_alignment_heatmap

alignment_matrix = npz["label_alignment_matrix"]
label_names = npz["label_names"].tolist()

plot_label_alignment_heatmap(
    alignment_matrix, label_names,
    show=True,
    save_path=FIGURES_DIR / "label_subspace_alignment",
)